# Практическая работа 2. Работа с LLM через API
**Студент:** Павлов Кирилл Кириллович  
**Группа:** 15.27Д-ИСТ05/26Б  
**Вариант:** 18 · 15 заявок · `rental_18.csv`

Цепочка: ChatPromptTemplate → GigaChat → StrOutputParser.
System задаёт роль и правила, Human содержит конкретную заявку.
В модель передаётся только text, без правильного ответа amount.


## Улучшение: few-shot + структурированный JSON
Модель извлекает count, count_adults, count_children, start_date, nights,
price_per_day и remarks. PydanticOutputParser проверяет типы, даты и сумму людей.
Эталон дополнительных полей размечен вручную; правила — в ANNOTATION_RULES.md.
Обе техники проверяются на одних и тех же 15 заявках; примеры few-shot
не взяты из проверочного варианта. Прогнозы не подгоняются под эталоны.

In [ ]:
import pandas as pd
from rental_llm import make_model, basic_chain, parse_count, safe_error
df = pd.read_csv('rental_18.csv', sep=';')
assert len(df) == 15
display(df)

In [ ]:
from rental_llm import run_experiment
status = run_experiment(advanced=True, with_metrics=True)
if 'structured' in status:
    display(pd.DataFrame(status['structured']['fields']).T)
    print(f"Средняя точность дополнительных полей: {status['structured']['mean_extra_accuracy']:.1%}")
    print(f"Общее количество, базовая техника: {status['basic']['accuracy']:.1%}")
    print(f"Общее количество, few-shot: {status['structured']['fields']['count']['accuracy']:.1%}")

При доступном API результаты сохраняются в `results/rental_with_results.csv`.
При проблеме подключения сохраняется `results/run_status.json`, а точность
не выдумывается. Для воспроизведения нужен локальный `.env` с GIGA_KEY.

`pd.to_numeric(..., errors='coerce')` превращает нечисловые результаты в NaN.
Сравнение с amount даёт число верных ответов; все 15 строк входят в знаменатель.
Ошибки запросов не удаляются из выборки.